IMPORT LIBRARY

In [40]:
import json
import os
import re
import numpy as np
import random
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tensorflow.keras.layers import Dense, Dropout, Input, Layer
from tensorflow.keras.models import Model
from tensorflow.keras.utils import register_keras_serializable
import tensorflow as tf
from tensorflow.keras import layers, Model, callbacks
import joblib
import pickle
import datetime


LOAD DATA

In [41]:
file_path = '../data/clean_recipes_5000.json'
with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

df = pd.DataFrame(data)
print("Jumlah resep:", len(df))
display(df.head())

Jumlah resep: 5000


,Title,Ingredients,Steps,Loves,URL,Category,Title Cleaned,Total Ingredients,Ingredients Cleaned,Total Steps,judul_bersih,bahan_bersih,Quality Score
0,Bakso Sapi (Pakai Blender),250 gram daging sapi--35 gram es batu (sekitar...,"1) Siapkan semua bahan, cincang daging sapi ke...",939,https://cookpad.com/id/resep/3431604-bakso-sap...,sapi,bakso sapi ( pakai blender ),8,"daging sapi , es batu cmx , putih telur , tepu...",10,bakso sapi pakai blender,250 gram daging sapi 35 gram es batu sekitar 4...,0.7997
1,Chilli Tuna Puff Kilat Super Yummy,1/2 pack Kulit puff instant saya merk.Edo--Bah...,1) Langkah \n1.Tumis chili tuna chunk dengan m...,516,https://cookpad.com/id/resep/4463240-chilli-tu...,ikan,chilli tuna puff kilat super yummy,9,"pack kulit puff instant merkedo , isian , kale...",3,chilli tuna puff kilat super yummy,1 2 pack kulit puff instant saya merk edo baha...,0.4689
2,Perkedel Tahu Simple,3 buah tahu petak--1 batang daun seledri--2 si...,1) Giling halus bawang merah+bawang putih+ mer...,481,https://cookpad.com/id/resep/4337985-perkedel-...,tahu,perkedel tahu simple,8,"tahu petak , batang daun seledri , bawang puti...",5,perkedel tahu simple,3 buah tahu petak 1 batang daun seledri 2 siun...,0.4827
3,Orek tempe basah bumbu ulek,1 papan tempe potong sesuai selera--3 buah cab...,1) Goreng tempe yg sdh di potong2 dlm minyak p...,452,https://cookpad.com/id/resep/3989566-orek-temp...,tempe,orek tempe basah bumbu ulek,12,"papan tempe potong , cabe ijo , kecap manis , ...",3,orek tempe basah bumbu ulek,1 papan tempe potong sesuai selera 3 buah cabe...,0.4223
4,Sop Iga Sapi Enaaak bangeet,"1 kg iga sapi, cuci bersih, tiriskan--Secukupn...","1) Didihkan secukupnya air, lalu masukan poton...",375,https://cookpad.com/id/resep/3310336-sop-iga-s...,sapi,sop iga sapi enaaak bangeet,22,"iga sapi , tiriskan , air didihkan utk rebusan...",4,sop iga sapi enaaak bangeet,"1 kg iga sapi, cuci bersih, tiriskan secukupny...",0.3550


EDA

In [42]:

df.info()
df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Title                5000 non-null   str    
 1   Ingredients          5000 non-null   str    
 2   Steps                5000 non-null   str    
 3   Loves                5000 non-null   int64  
 4   URL                  5000 non-null   str    
 5   Category             5000 non-null   str    
 6   Title Cleaned        5000 non-null   str    
 7   Total Ingredients    5000 non-null   int64  
 8   Ingredients Cleaned  5000 non-null   str    
 9   Total Steps          5000 non-null   int64  
 10  judul_bersih         5000 non-null   str    
 11  bahan_bersih         5000 non-null   str    
 12  Quality Score        5000 non-null   float64
dtypes: float64(1), int64(3), str(9)
memory usage: 507.9 KB


Title                  0
Ingredients            0
Steps                  0
Loves                  0
URL                    0
Category               0
Title Cleaned          0
Total Ingredients      0
Ingredients Cleaned    0
Total Steps            0
judul_bersih           0
bahan_bersih           0
Quality Score          0
dtype: int64

SET SEED

In [43]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [44]:
df.describe()

,Loves,Total Ingredients,Total Steps,Quality Score
count,5000.000000,5000.000000,5000.000000,5000.000000
mean,22.000000,12.507000,5.448800,0.300001
std,30.643339,4.615259,2.220172,0.068726
min,6.000000,3.000000,2.000000,0.084200
25%,9.000000,9.000000,4.000000,0.255600
50%,12.000000,12.000000,5.000000,0.293400
75%,25.000000,16.000000,7.000000,0.337675
max,939.000000,25.000000,23.000000,0.799700


In [45]:
# Hitung distribusi kategori
categories = [item.get('Category', 'Unknown') for item in data]
category_counts = Counter(categories)

print("Distribusi Kategori:")
for cat, count in category_counts.items():
    print(f"{cat}: {count}")

# Cek apakah seimbang
total = len(data)
print(f"\nTotal resep: {total}")
print("Kategori mayoritas:", category_counts.most_common(1)[0][0], "dengan", category_counts.most_common(1)[0][1], "resep")
print("Kategori minoritas:", category_counts.most_common()[-1][0], "dengan", category_counts.most_common()[-1][1], "resep")

# Hitung rasio
majority = category_counts.most_common(1)[0][1]
minority = category_counts.most_common()[-1][1]
ratio = majority / minority
print(f"Rasio mayoritas/minoritas: {ratio:.2f}")

if ratio > 2:
    print("Dataset tidak seimbang, mungkin ada bias terhadap kategori mayoritas.")
else:
    print("Dataset cukup seimbang.")

Distribusi Kategori:
sapi: 625
ikan: 625
tahu: 625
tempe: 625
kambing: 625
telur: 625
ayam: 625
udang: 625

Total resep: 5000
Kategori mayoritas: sapi dengan 625 resep
Kategori minoritas: udang dengan 625 resep
Rasio mayoritas/minoritas: 1.00
Dataset cukup seimbang.


In [46]:
df['ingredients_clean'] = df['Ingredients Cleaned'].fillna('')

PROPROCESSING

In [47]:
# Fitur numerik
num_features = ['Total Ingredients', 'Total Steps', 'Loves']
X_num = df[num_features].fillna(0).values

# Fitur teks (TF-IDF)
vectorizer = TfidfVectorizer(max_features=300, stop_words='english', min_df=2)
X_tfidf = vectorizer.fit_transform(df['ingredients_clean']).toarray()

# Gabungkan semua fitur (tanpa Category sebagai input, karena akan menjadi target)
X = np.hstack([X_num, X_tfidf])
print(f"Total fitur (X): {X.shape}")

# Target (Category, untuk klasifikasi)
# Gunakan LabelEncoder untuk mengubah kategori string menjadi angka integer
le_category = LabelEncoder()
y = le_category.fit_transform(df['Category'])
print(f"Target (y) shape: {y.shape}")
print(f"Mapping Kategori: {list(le_category.classes_)}")

Total fitur (X): (5000, 303)
Target (y) shape: (5000,)
Mapping Kategori: ['ayam', 'ikan', 'kambing', 'sapi', 'tahu', 'telur', 'tempe', 'udang']


SPLIT DATA

In [48]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"Train: {X_train.shape}, Validation: {X_val.shape}, Test: {X_test.shape}")

Train: (3500, 303), Validation: (750, 303), Test: (750, 303)


SCALING (STANDARDSCALER)

In [49]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Scaling selesai. Contoh nilai pertama train:\n", X_train_scaled[0])

Scaling selesai. Contoh nilai pertama train:
 [ 0.54604003  2.51281668 -0.17940142 -0.11083244 -0.10636446 -0.80717958
 -0.06945653 -0.17319073 -0.10737819  3.07948201 -0.22129659 -0.53890591
 -0.10938246 -0.11107417 -0.11551935 -0.09693344 -0.10181666 -0.07359292
 -0.34115882 -0.0897166  -0.43913614 -0.0867661  -0.16245547 -0.11374712
 -0.18389219 -0.12420255 -0.0840719  -0.07346439 -0.31734461 -0.09127572
 -0.09768282 -0.09555818 -0.08020979 -0.43630161 -0.08777513 -0.13993426
 -0.07915066 -0.16481919 -0.21305451  1.08269097 -0.09629752  0.50224372
 -0.17707462 -0.13933602 -0.09352833 -0.10397896 -0.32855731  0.07342613
 -0.13118887 -0.09719969 -0.2049185  -0.08208068 -0.26966878 -0.08556015
 -0.07714233 -0.07776723 -0.10606936 -0.19413745 -0.24685675 -0.49203466
  0.07753937 -0.08328325 -0.09327777 -0.10389216 -0.09770019 -0.13219662
 -0.08390265 -0.2105926  -0.12888214 -0.13795277 -0.09383944 -0.11117401
 -0.08467831 -0.129014   -0.14636375 -0.11011124 -0.08571534 -0.25403561
 -0.1

CUSTOM lAYER

In [50]:
@register_keras_serializable()
class IngredientsImportanceLayer(tf.keras.layers.Layer):
    def __init__(self, factor=1.2, **kwargs):
        super().__init__(**kwargs)
        self.factor = factor
    def call(self, inputs):
        return inputs * self.factor

CUSTOM LOSS

In [51]:
@register_keras_serializable()
def custom_recipe_loss(y_true, y_pred):
    loss = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
    return tf.reduce_mean(loss)

COSTUM CALLBACKS

In [52]:
class StopAtAccuracy(tf.keras.callbacks.Callback):
    def __init__(self, target=0.90):
        super().__init__()
        self.target = target
    def on_epoch_end(self, epoch, logs=None):
        val_acc = logs.get('val_accuracy')
        if val_acc and val_acc >= self.target:
            print(f"\n🎯 Target akurasi {self.target} tercapai di epoch {epoch+1}. Stop training.")
            self.model.stop_training = True


 BUILD MODEL (Functional API)

In [53]:

input_dim = X_train_scaled.shape[1] 
num_classes = len(le_category.classes_) 

input_layer = Input(shape=(input_dim,), name='input')
x = IngredientsImportanceLayer()(input_layer)

x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)

x = Dense(64, activation='relu')(x)
x = Dropout(0.3)(x)

output_layer = Dense(num_classes, activation='softmax', name='output')(x)

# Buat models
model = Model(inputs=input_layer, outputs=output_layer, name='MLP_recipes_model')
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()


Model: "MLP_recipes_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 303)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ingredients_importance_layer_2  │ (None, 303)            │             0 │
│ (IngredientsImportanceLayer)    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        38,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 8)              │           520 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 47,688 (186.28 KB)

 Trainable params: 47,688 (186.28 KB)

 Non-trainable params: 0 (0.00 B)

TENSORBOARD

In [54]:
log_dir = "logs/classification/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
writer = tf.summary.create_file_writer(log_dir)
print(f"TensorBoard logs: {log_dir}")


TensorBoard logs: logs/classification/20260603-225041


CUSTOM TRAINING LOOP (GradientTape) 

In [55]:
batch_size = 64
epochs = 100
patience = 10
best_val_acc = 0.0
wait = 0

# Dataset
train_dataset = tf.data.Dataset.from_tensor_slices((X_train_scaled, y_train))
train_dataset = train_dataset.shuffle(1024).batch(batch_size)

# Optimizer dan loss
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False)

for epoch in range(epochs):
    # ---- TRAINING ----
    train_loss = 0.0
    train_acc = 0.0
    num_batches = 0
    for batch_x, batch_y in train_dataset:
        with tf.GradientTape() as tape:
            logits = model(batch_x, training=True)
            loss = loss_fn(batch_y, logits)
        grads = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
        
        batch_acc = tf.reduce_mean(tf.cast(tf.equal(tf.argmax(logits, axis=1), batch_y), tf.float32))
        train_loss += loss.numpy()
        train_acc += batch_acc.numpy()
        num_batches += 1
    
    train_loss /= num_batches
    train_acc /= num_batches
    
    # ---- VALIDATION ----
    val_logits = model(X_val_scaled, training=False)
    val_loss = loss_fn(y_val, val_logits).numpy()
    val_acc = tf.reduce_mean(tf.cast(tf.equal(tf.argmax(val_logits, axis=1), y_val), tf.float32)).numpy()
    
    # ---- TENSORBOARD LOGGING ----
    with writer.as_default():
        tf.summary.scalar('loss/train', train_loss, step=epoch)
        tf.summary.scalar('accuracy/train', train_acc, step=epoch)
        tf.summary.scalar('loss/val', val_loss, step=epoch)
        tf.summary.scalar('accuracy/val', val_acc, step=epoch)
    
    # ---- PRINT PROGRESS ----
    print(f"Epoch {epoch+1:3d}/{epochs} | train_loss: {train_loss:.4f} | train_acc: {train_acc:.4f} | val_loss: {val_loss:.4f} | val_acc: {val_acc:.4f}")
    
    # ---- EARLY STOPPING ----
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        wait = 0
        model.save('best_model.keras')
        print(f"  --> Best model saved (val_acc={val_acc:.4f})")
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stopping. Best val_acc: {best_val_acc:.4f}")
            break

writer.close()

Epoch   1/100 | train_loss: 2.3736 | train_acc: 0.2028 | val_loss: 1.6723 | val_acc: 0.4413
  --> Best model saved (val_acc=0.4413)
Epoch   2/100 | train_loss: 1.7357 | train_acc: 0.3764 | val_loss: 1.3623 | val_acc: 0.6093
  --> Best model saved (val_acc=0.6093)
Epoch   3/100 | train_loss: 1.4068 | train_acc: 0.5032 | val_loss: 1.0411 | val_acc: 0.7187
  --> Best model saved (val_acc=0.7187)
Epoch   4/100 | train_loss: 1.0901 | train_acc: 0.6194 | val_loss: 0.7649 | val_acc: 0.7840
  --> Best model saved (val_acc=0.7840)
Epoch   5/100 | train_loss: 0.8838 | train_acc: 0.6937 | val_loss: 0.6187 | val_acc: 0.8133
  --> Best model saved (val_acc=0.8133)
Epoch   6/100 | train_loss: 0.7278 | train_acc: 0.7542 | val_loss: 0.5250 | val_acc: 0.8427
  --> Best model saved (val_acc=0.8427)
Epoch   7/100 | train_loss: 0.6318 | train_acc: 0.7900 | val_loss: 0.4666 | val_acc: 0.8547
  --> Best model saved (val_acc=0.8547)
Epoch   8/100 | train_loss: 0.5297 | train_acc: 0.8255 | val_loss: 0.4181 | 

EVALUASI MODEL

In [56]:
test_logits = model(X_test_scaled, training=False)
test_acc = tf.reduce_mean(tf.cast(tf.equal(tf.argmax(test_logits, axis=1), y_test), tf.float32)).numpy()
print(f"\nTest Accuracy: {test_acc:.4f}")
if test_acc >= 0.85:
    print("Target akurasi ≥ 85% tercapai!")
else:
    print(f"Akurasi masih {test_acc:.4f}")


Test Accuracy: 0.9067
Target akurasi ≥ 85% tercapai!


SAVE MODEL

In [57]:
models_dir = "saved_models" 
os.makedirs(models_dir, exist_ok=True)

# Simpan model
model_path = os.path.join(models_dir, "MLP_recipes_model.keras")
model.save(model_path)

# Simpan vectorizer
vectorizer_path = os.path.join(models_dir, "tf_idf_vectorizer.pkl")
with open(vectorizer_path, "wb") as f:
    pickle.dump(vectorizer, f)

# Simpan label encoder
label_encoder_path = os.path.join(models_dir, "label_encoder.pkl")
with open(label_encoder_path, "wb") as f:
    pickle.dump(le_category, f)

print(f"Model disimpan: {model_path}")
print(f"Vectorizer disimpan: {vectorizer_path}")
print(f"Label Encoder disimpan: {label_encoder_path}")

Model disimpan: saved_models\MLP_recipes_model.keras
Vectorizer disimpan: saved_models\tf_idf_vectorizer.pkl
Label Encoder disimpan: saved_models\label_encoder.pkl


INFERENCE MODEL

In [58]:
# Cetak seluruh daftar kategori
print("Daftar kategori (indeks -> kategori):")
for i, cat in enumerate(le_category.classes_):
    print(f"  {i} -> {cat}")

Daftar kategori (indeks -> kategori):
  0 -> ayam
  1 -> ikan
  2 -> kambing
  3 -> sapi
  4 -> tahu
  5 -> telur
  6 -> tempe
  7 -> udang


In [59]:
model_path = 'saved_models/MLP_recipes_model.keras'
encoder_path = 'saved_models/label_encoder.pkl'

# Load model dan encoder
model = tf.keras.models.load_model(model_path)
with open(encoder_path, 'rb') as f:
    le_category = pickle.load(f)

print("Model dan label encoder berhasil dimuat.")
print(f"Model input shape: {model.input_shape}")
print(f"Jumlah kelas: {len(le_category.classes_)}")

# Jika tidak, buat input acak sesuai dimensi model (hanya untuk demonstrasi)
input_dim = model.input_shape[1]
dummy_input = np.random.rand(1, input_dim).astype(np.float32)
actual_label = None  # tidak ada label asli

print(f"Input shape: {dummy_input.shape}")


# LAKUKAN PREDIKSI

predictions = model.predict(dummy_input)
predicted_class_index = np.argmax(predictions, axis=1)[0]
predicted_category = le_category.inverse_transform([predicted_class_index])[0]

print(f"\n--- HASIL PREDIKSI ---")
print(f"Probabilitas per kelas: {predictions[0]}")
print(f"Prediksi kelas index: {predicted_class_index}")
print(f"Prediksi kategori: {predicted_category}")


c:\Users\yohan\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\saving\saving_lib.py:801: UserWarning: Skipping variable loading for optimizer 'adam', because it has 14 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model dan label encoder berhasil dimuat.
Model input shape: (None, 303)
Jumlah kelas: 8
Input shape: (1, 303)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 358ms/step

--- HASIL PREDIKSI ---
Probabilitas per kelas: [0.06678512 0.4623736  0.00877291 0.04279538 0.38226795 0.01051877
 0.00500191 0.02148422]
Prediksi kelas index: 1
Prediksi kategori: ikan


In [60]:
print(f"Indeks {predicted_class_index} adalah kategori: {le_category.classes_[predicted_class_index]}")

Indeks 1 adalah kategori: ikan


In [ ]:
"""
Jalankan TensorBoard dengan perintah berikut di terminal:

cd "Quest\Path AI\DEEP LEARNING"
tensorboard --logdir logs --port 6007

Lalu buka:
http://localhost:6007

Catatan:
- Grafik quality_score berasal dari logs/quality_score
"""

'\nJalankan TensorBoard dengan perintah berikut di terminal:\n\n.venv\\Scripts\x07ctivate\ncd "Quest\\Path AI\\DEEP LEARNING"\ntensorboard --logdir logs --port 6007\n\nLalu buka:\nhttp://localhost:6007\n\nCatatan:\n- Grafik quality_score berasal dari logs/quality_score\n'